In [1]:
from pyspark.sql import SparkSession

In [2]:
%load_ext autoreload
%autoreload 2


In [3]:
from pathlib import Path
import sys

# Get the absolute path of the parent folder
parent_dir = str(Path().resolve().parent)

if parent_dir not in sys.path:
    sys.path.append(parent_dir)
import config

In [4]:
spark = SparkSession.builder.appName('StoreCast_Bronze_Ingestion').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/31 09:56:29 WARN Utils: Your hostname, LAPTOP-L5F30S0E, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/31 09:56:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/31 09:56:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
df_sales = spark.read.csv(str(config.RAW_SALES_PATH), header=True, inferSchema=True)
df_features = spark.read.csv(str(config.RAW_FEATURES_PATH), header=True, inferSchema=True)
df_stores = spark.read.csv(str(config.RAW_STORES_PATH), header=True, inferSchema=True)

In [6]:
df_sales.show(5)
df_features.show(5)
df_stores.show(5)

+-----+----+----------+------------+---------+
|Store|Dept|      Date|Weekly_Sales|IsHoliday|
+-----+----+----------+------------+---------+
|    1|   1|05/02/2010|     24924.5|    false|
|    1|   1|12/02/2010|    46039.49|     true|
|    1|   1|19/02/2010|    41595.55|    false|
|    1|   1|26/02/2010|    19403.54|    false|
|    1|   1|05/03/2010|     21827.9|    false|
+-----+----+----------+------------+---------+
only showing top 5 rows
+-----+----------+-----------+----------+---------+---------+---------+---------+---------+-----------+------------+---------+
|Store|      Date|Temperature|Fuel_Price|MarkDown1|MarkDown2|MarkDown3|MarkDown4|MarkDown5|        CPI|Unemployment|IsHoliday|
+-----+----------+-----------+----------+---------+---------+---------+---------+---------+-----------+------------+---------+
|    1|05/02/2010|      42.31|     2.572|       NA|       NA|       NA|       NA|       NA|211.0963582|       8.106|    false|
|    1|12/02/2010|      38.51|     2.548|    

In [11]:
df_sales.schema
df_stores.schema 
df_features.schema

StructType([StructField('Store', IntegerType(), True), StructField('Date', StringType(), True), StructField('Temperature', DoubleType(), True), StructField('Fuel_Price', DoubleType(), True), StructField('MarkDown1', StringType(), True), StructField('MarkDown2', StringType(), True), StructField('MarkDown3', StringType(), True), StructField('MarkDown4', StringType(), True), StructField('MarkDown5', StringType(), True), StructField('CPI', StringType(), True), StructField('Unemployment', StringType(), True), StructField('IsHoliday', BooleanType(), True)])

In [10]:
df_sales.write.mode('overwrite').partitionBy('Store').parquet(str(config.BRONZE_SALES_PATH))
df_features.write.mode('overwrite').parquet(str(config.BRONZE_FEATURES_PATH))
df_stores.write.mode('overwrite').parquet(str(config.BRONZE_STORES_PATH))

In [9]:
df_sales.schema

StructType([StructField('Store', IntegerType(), True), StructField('Dept', IntegerType(), True), StructField('Date', StringType(), True), StructField('Weekly_Sales', DoubleType(), True), StructField('IsHoliday', BooleanType(), True)])